# FMCG Sales Performance Analysis
## Data Analyst Portfolio Project

**Author:** [Your Name]  
**Tools:** Python (Pandas, NumPy, Matplotlib, Seaborn), SQL, Tableau

---

### 📌 Project Background
Sebagai data analyst di perusahaan FMCG, salah satu tanggung jawab utama adalah memantau **sales performance** lintas cabang, principal, dan produk. Analisis ini bertujuan untuk:

1. Memberikan gambaran kondisi penjualan keseluruhan (2023–2024)
2. Mengidentifikasi cabang & produk top performer dan under performer
3. Mengukur achievement vs target
4. Memberikan rekomendasi berbasis data untuk meningkatkan profitabilitas

### 📋 Business Questions
1. Bagaimana tren penjualan tahunan & bulanan?
2. Cabang & region mana yang paling profitable?
3. Principal mana yang berkontribusi terbesar?
4. Produk apa yang menjadi best-seller dan mana yang slow-moving?
5. Apakah target 2024 tercapai?
6. Pola seasonality apa yang terlihat (Ramadan, akhir tahun)?

## 1. Data Preparation
Import library, load data, dan inspeksi awal.

In [ ]:
# Library untuk manipulasi data
import pandas as pd
import numpy as np

# Library untuk visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Pengaturan tampilan agar output lebih rapi
pd.set_option('display.max_columns', None)        # tampilkan semua kolom
pd.set_option('display.float_format', '{:,.2f}'.format)  # format ribuan
sns.set_style('whitegrid')                         # background plot dengan grid
plt.rcParams['figure.figsize'] = (12, 6)           # ukuran default plot
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully ✓')

In [ ]:
# Load 4 file CSV dari folder data
# pd.read_csv() membaca file CSV dan mengubahnya menjadi DataFrame
df_sales      = pd.read_csv('../data/sales_transactions.csv')
df_branches   = pd.read_csv('../data/branches.csv')
df_products   = pd.read_csv('../data/products.csv')
df_principals = pd.read_csv('../data/principals.csv')

print(f'Sales transactions : {df_sales.shape[0]:,} rows × {df_sales.shape[1]} cols')
print(f'Branches           : {df_branches.shape[0]:,} rows')
print(f'Products           : {df_products.shape[0]:,} rows')
print(f'Principals         : {df_principals.shape[0]:,} rows')

In [ ]:
# .head() menampilkan 5 baris pertama untuk inspeksi cepat
df_sales.head()

In [ ]:
# .info() menampilkan tipe data, jumlah non-null per kolom, dan memory usage
# Berguna untuk mendeteksi NULL dan tipe data yang salah (mis. tanggal sebagai string)
df_sales.info()

In [ ]:
# .describe() memberikan statistik deskriptif: mean, std, min, max, quartile
# Sangat berguna untuk spot anomali (mis. min negatif, max yang tidak masuk akal)
df_sales.describe()

## 2. Data Cleaning

Dari `.describe()` di atas terlihat ada anomali:
- `quantity` punya nilai **minimum negatif** → tidak masuk akal
- Ada **NULL values** di kolom `quantity` & `revenue`
- Mungkin ada **whitespace** di `branch_id`
- Mungkin ada **duplikat** transaksi

In [ ]:
# === STEP 1: Cek missing values ===
# .isnull() ubah setiap nilai jadi True/False, .sum() hitung True per kolom
missing = df_sales.isnull().sum()
print('Missing values per kolom:')
print(missing[missing > 0])

In [ ]:
# === STEP 2: Cek duplikat ===
# .duplicated() mark baris duplikat berdasarkan subset kolom
dup_count = df_sales.duplicated(subset=['transaction_id']).sum()
print(f'Jumlah baris duplikat berdasarkan transaction_id: {dup_count}')

In [ ]:
# === STEP 3: Cek nilai negatif ===
neg_qty = (df_sales['quantity'] < 0).sum()
neg_rev = (df_sales['revenue'] < 0).sum()
print(f'Quantity negatif: {neg_qty} | Revenue negatif: {neg_rev}')

In [ ]:
# === STEP 4: Cek whitespace di branch_id ===
# .str.strip() hilangkan spasi, bandingkan dengan original
ws_count = (df_sales['branch_id'] != df_sales['branch_id'].str.strip()).sum()
print(f'Branch_id dengan whitespace: {ws_count}')

In [ ]:
# === CLEANING PIPELINE ===
# Buat copy agar data asli tidak rusak (.copy() penting!)
df_clean = df_sales.copy()

# 1. Drop NULL di kolom kritis
df_clean = df_clean.dropna(subset=['quantity', 'revenue'])

# 2. Drop nilai negatif
df_clean = df_clean[(df_clean['quantity'] > 0) & (df_clean['revenue'] > 0)]

# 3. Strip whitespace
df_clean['branch_id'] = df_clean['branch_id'].str.strip()

# 4. Drop duplikat (keep='first' = simpan baris pertama)
df_clean = df_clean.drop_duplicates(subset=['transaction_id'], keep='first')

# 5. Convert transaction_date ke datetime untuk operasi tanggal
df_clean['transaction_date'] = pd.to_datetime(df_clean['transaction_date'])

# 6. Tambah kolom turunan (feature engineering)
df_clean['gross_profit'] = df_clean['revenue'] - df_clean['cost']
df_clean['margin_pct']   = (df_clean['gross_profit'] / df_clean['revenue']) * 100
df_clean['year']         = df_clean['transaction_date'].dt.year
df_clean['month']        = df_clean['transaction_date'].dt.month
df_clean['year_month']   = df_clean['transaction_date'].dt.to_period('M').astype(str)
df_clean['quarter']      = df_clean['transaction_date'].dt.quarter
df_clean['day_of_week']  = df_clean['transaction_date'].dt.day_name()

print(f'Sebelum cleaning : {len(df_sales):,} rows')
print(f'Setelah cleaning : {len(df_clean):,} rows')
print(f'Rows dibuang     : {len(df_sales) - len(df_clean):,} ({(len(df_sales)-len(df_clean))/len(df_sales)*100:.2f}%)')

## 3. Data Modification (Joining Tables)
Gabungkan tabel transaksi dengan master data agar bisa analisis berdasarkan nama (bukan sekedar ID).

In [ ]:
# pd.merge() = SQL JOIN. how='left' = LEFT JOIN.
# Step 1: Sales + Branches
df = df_clean.merge(df_branches, on='branch_id', how='left')

# Step 2: + Products
df = df.merge(df_products, on='product_id', how='left')

# Step 3: + Principals
df = df.merge(df_principals, on='principal_id', how='left',
              suffixes=('_product', '_principal'))   # cegah konflik nama kolom

print(f'Final dataset: {df.shape[0]:,} rows × {df.shape[1]} cols')
df.head(3)

## 4. Exploratory Data Analysis (EDA) & Visualization

### 4.1 Overall KPI Snapshot

In [ ]:
# Hitung KPI utama dan tampilkan dalam format yang mudah dibaca
total_revenue = df['revenue'].sum()
total_profit  = df['gross_profit'].sum()
total_units   = df['quantity'].sum()
total_trans   = df['transaction_id'].nunique()
avg_margin    = (total_profit / total_revenue) * 100

print('=' * 50)
print('  KPI SUMMARY (Jan 2023 – Dec 2024)')
print('=' * 50)
print(f'  Total Revenue    : Rp {total_revenue:>20,.0f}')
print(f'  Total Profit     : Rp {total_profit:>20,.0f}')
print(f'  Gross Margin     :    {avg_margin:>19,.2f} %')
print(f'  Units Sold       :    {total_units:>19,.0f}')
print(f'  Transactions     :    {total_trans:>19,.0f}')

### 4.2 Monthly Revenue Trend

In [ ]:
# .groupby() = SQL GROUP BY. Hitung total revenue per year_month.
monthly = df.groupby('year_month').agg(
    revenue=('revenue', 'sum'),
    profit=('gross_profit', 'sum'),
    transactions=('transaction_id', 'nunique')
).reset_index()

# Hitung Month-over-Month growth dengan .pct_change()
monthly['mom_growth_pct'] = monthly['revenue'].pct_change() * 100

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['year_month'], monthly['revenue']/1e9,
        marker='o', linewidth=2, color='#2C5F2D')
ax.fill_between(monthly['year_month'], monthly['revenue']/1e9,
                alpha=0.2, color='#2C5F2D')
ax.set_title('Monthly Revenue Trend (2023–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue (Rp Miliar)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../docs/01_monthly_trend.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n💡 Insight: Terlihat lonjakan revenue di Maret–April (Ramadan boost) dan Desember (akhir tahun).')

### 4.3 Branch Performance Ranking

In [ ]:
# Aggregate per cabang
branch_perf = df.groupby('branch_name').agg(
    revenue=('revenue', 'sum'),
    profit=('gross_profit', 'sum'),
    transactions=('transaction_id', 'nunique')
).sort_values('revenue', ascending=True).reset_index()       # ascending utk barh chart

branch_perf['margin_pct'] = (branch_perf['profit'] / branch_perf['revenue']) * 100

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#B85042' if i == len(branch_perf)-1 else '#A7BEAE' for i in range(len(branch_perf))]
bars = ax.barh(branch_perf['branch_name'], branch_perf['revenue']/1e9, color=colors)

# Annotasi nilai pada bar
for bar, val in zip(bars, branch_perf['revenue']/1e9):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}B', va='center', fontsize=10)

ax.set_title('Revenue by Branch', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (Rp Miliar)')
plt.tight_layout()
plt.savefig('../docs/02_branch_revenue.png', dpi=120, bbox_inches='tight')
plt.show()

branch_perf.sort_values('revenue', ascending=False)

### 4.4 Principal Contribution

In [ ]:
principal_perf = df.groupby('principal_name').agg(
    revenue=('revenue', 'sum'),
    profit=('gross_profit', 'sum')
).sort_values('revenue', ascending=False).reset_index()

principal_perf['share_pct'] = (principal_perf['revenue'] / principal_perf['revenue'].sum()) * 100

# Donut chart
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#1E2761', '#065A82', '#1C7293', '#84B59F', '#A7BEAE']
wedges, texts, autotexts = ax.pie(
    principal_perf['revenue'],
    labels=principal_perf['principal_name'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.4)         # bikin donut (lubang di tengah)
)
for t in autotexts:
    t.set_color('white'); t.set_fontweight('bold')

ax.set_title('Revenue Share by Principal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/03_principal_share.png', dpi=120, bbox_inches='tight')
plt.show()

principal_perf

### 4.5 Top 10 Best-Selling Products

In [ ]:
top_products = df.groupby('product_name').agg(
    units=('quantity', 'sum'),
    revenue=('revenue', 'sum'),
    profit=('gross_profit', 'sum')
).sort_values('revenue', ascending=False).head(10).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_products['product_name'][::-1], top_products['revenue'][::-1]/1e9,
        color='#065A82')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue (Rp Miliar)')
plt.tight_layout()
plt.savefig('../docs/04_top_products.png', dpi=120, bbox_inches='tight')
plt.show()

top_products

### 4.6 Heatmap: Branch × Principal
Mengetahui kombinasi cabang & principal mana yang paling kuat.

In [ ]:
# pivot_table = SQL pivot. Index = baris, columns = kolom, values = sel
pivot = df.pivot_table(
    index='branch_name',
    columns='principal_name',
    values='revenue',
    aggfunc='sum'
) / 1e9   # ubah ke miliar

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu',
            cbar_kws={'label': 'Revenue (Rp Miliar)'}, ax=ax)
ax.set_title('Revenue Heatmap: Branch × Principal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/05_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.7 Year-over-Year Growth

In [ ]:
# Pivot revenue per year per branch lalu hitung YoY growth
yoy = df.pivot_table(index='branch_name', columns='year',
                     values='revenue', aggfunc='sum') / 1e9
yoy['yoy_growth_pct'] = ((yoy[2024] - yoy[2023]) / yoy[2023]) * 100
yoy = yoy.sort_values('yoy_growth_pct', ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#B85042' if v < 10 else '#2C5F2D' for v in yoy['yoy_growth_pct']]
bars = ax.barh(yoy.index, yoy['yoy_growth_pct'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, yoy['yoy_growth_pct']):
    ax.text(val + 0.3 if val > 0 else val - 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%', va='center',
            ha='left' if val > 0 else 'right', fontsize=10)
ax.set_title('YoY Revenue Growth by Branch (2024 vs 2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Growth (%)')
plt.tight_layout()
plt.savefig('../docs/06_yoy_growth.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.8 Achievement vs Target
Asumsi: target 2024 = realisasi 2023 + 15% growth.

In [ ]:
# Hitung target & actual per produk
qty_2023 = df[df['year']==2023].groupby('product_name')['quantity'].sum()
qty_2024 = df[df['year']==2024].groupby('product_name')['quantity'].sum()

achievement = pd.DataFrame({
    'target_2024': qty_2023 * 1.15,
    'actual_2024': qty_2024
}).dropna()
achievement['achievement_pct'] = (achievement['actual_2024'] / achievement['target_2024']) * 100
achievement['status'] = pd.cut(
    achievement['achievement_pct'],
    bins=[0, 80, 100, float('inf')],
    labels=['Under Target', 'Almost There', 'Achieved']
)

# Status counts
status_count = achievement['status'].value_counts()
print('Distribusi achievement status:')
print(status_count)
print(f"\nTop 5 over-achiever:")
print(achievement.nlargest(5, 'achievement_pct')[['target_2024','actual_2024','achievement_pct']])
print(f"\nBottom 5 under-achiever:")
print(achievement.nsmallest(5, 'achievement_pct')[['target_2024','actual_2024','achievement_pct']])

## 5. Key Findings & Recommendations

### 🔍 Key Findings
1. **Total Revenue 2023–2024**: ~Rp 15.6 Miliar dengan margin sehat ~14%
2. **Top Performer Cabang**: Jakarta Pusat & Surabaya menyumbang >30% revenue
3. **Principal Dominant**: Nestle Indonesia (~46% share) → risiko ketergantungan tinggi
4. **Best Seller**: Dancow Fortigro & Nescafe Classic (high price × high volume)
5. **Seasonality**: Lonjakan signifikan di Maret–April (Ramadan) dan Desember
6. **YoY 2024**: Sebagian besar cabang growth >10%, sesuai target nasional

### 💡 Recommendations
1. **Diversifikasi principal** — kurangi over-reliance pada Nestle dengan ekspansi Mayora & Indofood
2. **Boost cabang Sulawesi & Sumatera** — gap revenue masih besar dibanding Jabodetabek
3. **Ramadan campaign** — alokasi inventory & promosi lebih agresif di Q1
4. **Review pricing & margin** — beberapa produk volume tinggi tapi margin rendah
5. **Real-time dashboard** — implementasikan Tableau dashboard untuk monitoring harian

## 6. Export untuk Tableau
Simpan data hasil cleaning dalam format yang siap diimpor ke Tableau.

In [ ]:
# Export final dataset untuk Tableau
df.to_csv('../tableau/sales_clean_for_tableau.csv', index=False)
print(f'Exported: ../tableau/sales_clean_for_tableau.csv ({len(df):,} rows)')